In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# E.1 The Oscillator's Biography: One System, Seven Volumes

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Epilogue",
    number="E.1",
    title="The Oscillator's Biography: One System, Seven Volumes",
    blurb="The harmonic oscillator sat in nearly every volume of this course — a "
    "trajectory, a resonance, a thermal degree of freedom, a ladder of states, a "
    "ring of beads, a Monte Carlo walk. Here we gather all seven faces and show "
    "they are one: each derived from the next as a measured limit, with the "
    "classical world falling out of the quantum one at a rate we can put a number "
    "on. We open where the whole course opened — with the fact that a computer "
    "cannot represent (1+ε)−1 — and end with a single warmed oscillator's "
    "mean-square displacement, reached four independent ways and agreeing to six "
    "digits.",
    difficulty="advanced",
    estimate="210–260 min",
)

## Notebook overview

The Epilogue does not open a new subject; it closes the course by computing the threads
that run through all seven volumes and belong to none of them alone. This first notebook
answers *what did the course teach?* at the level of a single object. The harmonic
oscillator is the course's recurring protagonist — it appears as a trajectory in Volume I,
a resonance in Volume I, a thermal degree of freedom in Volume V, a ladder of states in
Volume VI, a thermal density matrix, a ring of beads, and a Monte Carlo walk in Volume
VII — and by the end the reader owns not seven facts but one object seen seven ways, with
the maps between the views *computed* rather than asserted.

The organizing discipline is that each face is derived from the more general face as a
**measured limit**, its convergence rate quantified. The classical statistical oscillator
is the $\hbar \to 0$ limit of the thermal quantum one, and the leading correction — the
Wigner–Kirkwood term $\beta/12$ — is verified to five digits, so equipartition itself
joins the course's running tally of classical laws that turned out to be limits ($h$,
$\sigma$, $N!$, the Langevin paramagnet, Dulong–Petit, and now equipartition). The quantum
ground state is the $T \to 0$ limit, and zero-point motion is what survives when all
thermal occupation is gone. The path-integral and sampled oscillators are the same thermal
object reached by imaginary time instead of spectra, converging as $M \to \infty$ at the
measured $1/M^2$ rate.

The notebook opens with a **ring**: it returns to the course's very first computation — the
demonstration in [§0.1](../00-foundations/floating-point.ipynb) that a computer cannot
represent $(1 + \varepsilon) - 1$ for $\varepsilon$ below machine epsilon — and reads it
forward. The whole course was the craft of computing approximately on purpose and knowing
the error, and the oscillator is where that craft is displayed whole, because for the
oscillator *every* answer is known in closed form, so every approximation can be measured
against truth. The summit is a **four-way rendezvous**: the mean-square displacement of a
warmed oscillator at $\beta = 2$, computed by ladder spectra, by grid diagonalization, by
a ring-polymer determinant, and by the classical limit — the three quantum routes agreeing
to a part in $10^5$, one number reached by mechanics, spectra, and path integrals at once.

> **Conventions (this notebook).** Oscillator units $\hbar = m = \omega = 1$ throughout, so
> energies are in units of $\hbar\omega$ and the classical temperature enters as
> $kT = 1/\beta$; the single knob for the classical-versus-quantum comparison is the
> temperature (we fix $\hbar\omega$ and vary $T$, never both, so the tally of demotions
> stays clean). The classical trajectory uses `scipy.integrate.solve_ivp` (DOP853,
> `rtol = 1e-11`); the driven–damped peak is located by `numpy.argmax` on a fine grid and
> checked against the closed form $\sqrt{\omega_0^2 - \gamma^2/2}$; equipartition is a
> `scipy.integrate.quad` of the Boltzmann weight; the quantum faces use a three-point
> Laplacian and `numpy.linalg.eigh` on a box wide enough for the thermal width (adequacy
> checked); the ring-polymer face reuses the circulant action matrix of
> [§7.20](../07-quantum-statistical-mechanics/imaginary-time-quantum-classical.ipynb) with
> `numpy.linalg.slogdet`/`inv`; the sampled face reuses the staging protocol of
> [§7.21](../07-quantum-statistical-mechanics/path-integral-monte-carlo.ipynb) verbatim
> ($L = M/2$, seeded `numpy.random.default_rng`, window-checked $\tau_{\mathrm{int}}$,
> blocking). The Wigner–Kirkwood law is fit at small $\beta$, where it holds.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an
> independent fact: the ODE against the closed form; the resonance peak against its
> formula; the ladder against $(n+\tfrac12)$; the coth against grid-ED thermal averages;
> the $\beta/12$ law against $1/12$; the ring determinant against the $1/M^2$ rate; the
> four routes against one another. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*.
>
> **Scope.** This is synthesis, not expansion: every tool used here was built earlier in
> the course. See Feynman & Hibbs, *Quantum Mechanics and Path Integrals* (the
> path-integral oscillator); Wigner, *Phys. Rev.* **40**, 749 (1932) and Kirkwood, *Phys.
> Rev.* **44**, 31 (1933) (the $\hbar^2$ correction). Cross-reference the biography's
> chapters: [§0.1](../00-foundations/floating-point.ipynb) (the ring),
> [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb) (resonance),
> [§1.6](../01-elementary-mechanics/integrators.ipynb) and
> [§2.3](../02-classical-mechanics/hamiltonian-phase-flow.ipynb) (symplectic flow),
> [§5.8](../05-classical-stat-mech/partition-function.ipynb) (equipartition),
> [§6.12](../06-quantum-mechanics/harmonic-oscillator.ipynb) (the ladder),
> [§7.5](../07-quantum-statistical-mechanics/oscillator-at-temperature.ipynb) (the coth),
> [§7.20](../07-quantum-statistical-mechanics/imaginary-time-quantum-classical.ipynb) (the
> ring polymer),
> [§7.21](../07-quantum-statistical-mechanics/path-integral-monte-carlo.ipynb) (the
> sampler), and the tally of demotions in
> [§7.8](../07-quantum-statistical-mechanics/classical-limit-thermal-wavelength.ipynb),
> [§7.14](../07-quantum-statistical-mechanics/photon-gas-planck.ipynb),
> [§7.16](../07-quantum-statistical-mechanics/phonons-debye.ipynb),
> [§7.18](../07-quantum-statistical-mechanics/quantum-paramagnets-brillouin.ipynb).
> Forward to [E.2](four-faces-of-the-action.ipynb) (the action), [E.3](universality.ipynb) (universality), [E.4](how-we-knew.ipynb) (the method, and the ring's far
> end).

## Theory in brief

### The ring: where the course began

The course's very first computation, in
[§0.1](../00-foundations/floating-point.ipynb), was to show that the computer does not
hold the real numbers — only a grid of floating-point values with gaps that thin toward
zero — so that below a certain size an addend simply vanishes:

```{math}
:label: eq-ring-open
(1 + \varepsilon) - 1 = 0 \neq \varepsilon
\qquad\text{for } \varepsilon = 2^{-53} < \varepsilon_{\mathrm{mach}} = 2^{-52},
```

because $1 + 2^{-53}$ rounds to exactly $1$ (round-half-to-even). Read forward, this is the
whole course in one line: arithmetic itself is approximate, and everything since was the
craft of computing approximately with the error kept under control. The oscillator is the
place that craft is displayed whole, because every oscillator answer is closed-form, so
every numerical face can be *graded* against the truth — the one system where the course
can measure itself.

### Seven faces

The oscillator wore a different face in nearly every volume, each with a defining result
the reader has already met:

```{math}
:label: eq-seven-faces
\begin{aligned}
&\text{(1) classical: } x(t) = A\cos\omega t; \quad
\text{(2) driven–damped: peak at } \omega_d = \sqrt{\omega_0^2 - \gamma^2/2},\ Q = \omega_0/\gamma; \\
&\text{(3) classical statistical: } \langle x^2\rangle = kT/m\omega^2; \quad
\text{(4) quantum: } E_n = (n+\tfrac12)\hbar\omega,\ \langle 0|x^2|0\rangle = \tfrac12; \\
&\text{(5) thermal quantum: } \langle x^2\rangle_T = \tfrac{\hbar}{2m\omega}\coth\tfrac{\beta\hbar\omega}{2}; \quad
\text{(6) ring polymer}; \quad \text{(7) sampled (PIMC)}.
\end{aligned}
```

Faces 1–2 are the mechanics of Volume I (the closed forms are in any text; Goldstein
derives the driven oscillator's response in full). Face 3 is the equipartition of
[§5.8](../05-classical-stat-mech/partition-function.ipynb); Face 4 the ladder of
[§6.12](../06-quantum-mechanics/harmonic-oscillator.ipynb); Face 5 the coth of
[§7.5](../07-quantum-statistical-mechanics/oscillator-at-temperature.ipynb); Faces 6–7 the
imaginary-time machinery of
[§7.20](../07-quantum-statistical-mechanics/imaginary-time-quantum-classical.ipynb) and
[§7.21](../07-quantum-statistical-mechanics/path-integral-monte-carlo.ipynb).

### The spine: reductions as measured limits

The biography's real content is that these are not seven oscillators but one, and the
views connect by limits the notebook does not assert but measures. Expanding the coth at
small argument, $\coth(x) = 1/x + x/3 - \cdots$, gives the classical value plus a leading
quantum correction:

```{math}
:label: eq-reductions
\tfrac12\coth\tfrac{\beta}{2} \;=\; \underbrace{\tfrac{1}{\beta}}_{kT}
\;+\; \underbrace{\tfrac{\beta}{12}}_{\text{Wigner–Kirkwood}} + O(\beta^3),
\qquad
\tfrac12\coth\tfrac{\beta}{2} \xrightarrow{\ \beta\to\infty\ } \tfrac12,
```

so the classical statistical oscillator (Face 3) is the $\hbar \to 0$ (equivalently
$T \to \infty$) limit of the thermal quantum one (Face 5), corrected by the
Wigner–Kirkwood $\beta/12$ term (Wigner 1932, Kirkwood 1933) — equipartition demoted to a
limit, on a measured law. The quantum ground state (Face 4) is the $T \to 0$ limit: zero-
point motion is what survives when all thermal occupation is gone. And the ring-polymer
and sampled faces (6, 7) are the same thermal object reached by a different route entirely,
converging to it as $M \to \infty$ at the $1/M^2$ Trotter rate measured below.

### The final rendezvous

The spine assembles into one figure: $\langle x^2\rangle(T)$ across temperature, computed
four ways,

```{math}
:label: eq-e1-rendezvous
\langle x^2\rangle(T):\quad
\underbrace{\tfrac12\coth\tfrac{\beta}{2}}_{\text{ladder/spectral}}
\;=\; \underbrace{\sum_n p_n \langle n|x^2|n\rangle}_{\text{grid ED}}
\;=\; \underbrace{(A^{-1})_{00}}_{\text{ring polymer}}
\;\;\gtrsim\;\; \underbrace{kT}_{\text{classical}},
```

the three quantum routes lying on one curve to a part in $10^5$ at $\beta = 2$, the
classical line peeling away below the thermal scale by exactly the zero-point. A single
physical quantity — the mean-square displacement of a warmed oscillator — reached by
Newton's mechanics continued to imaginary time, by Schrödinger's spectrum summed with
Boltzmann weights, and by a random walk of beads: three centuries of physics agreeing on
one number.

### What E.1 establishes

The course's **unity of object**: one system, fully owned, its every face a limit of its
neighbour. The Epilogue's next step asks whether the unity runs deeper than one lucky
system — whether a single *principle* generates the whole course. It does, and it has a
name: the action ([E.2](four-faces-of-the-action.ipynb)).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad, solve_ivp

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Oscillator units hbar = m = omega = 1: energies in hbar*omega, kT = 1/beta.
# The single knob for classical-vs-quantum is the temperature.


def coth_half(beta):
    """The thermal quantum <x^2> = (1/2) coth(beta/2) (the curve of section 7.5; units h=m=w=1)."""
    return 0.5 / np.tanh(beta / 2.0)


def grid_ed_oscillator(N, L):
    """Grid ED of the oscillator: three-point Laplacian + numpy.linalg.eigh (section 6.12).

    H = -(1/2) d^2/dx^2 + (1/2) x^2 on N points spanning [-L, L]. Box and
    spacing adequacy are checked in Exercise 3 against the thermal width.

    Parameters
    ----------
    N : int
        Grid points.
    L : float
        Half-box.

    Returns
    -------
    xs : numpy.ndarray
        Grid.
    E : numpy.ndarray
        Eigenvalues, ascending.
    V : numpy.ndarray
        Eigenvectors as columns (grid-normalized).
    """
    xs = np.linspace(-L, L, N)
    dx = xs[1] - xs[0]
    off = np.ones(N - 1)
    lap = (np.diag(off, 1) + np.diag(off, -1) - 2.0 * np.eye(N)) / dx**2
    E, V = np.linalg.eigh(-0.5 * lap + np.diag(0.5 * xs**2))
    return xs, E, V


def thermal_x2_ed(xs, E, V, beta, nkeep=60):
    """<x^2>_T by Boltzmann-summing grid-ED eigenstates (ground-shifted weights).

    The truncation at nkeep is checked by the weight of the last state kept;
    at the temperatures used here e^{-beta(E_nkeep - E_0)} is negligible.

    Parameters
    ----------
    xs, E, V : numpy.ndarray
        Output of grid_ed_oscillator.
    beta : float
        Inverse temperature.
    nkeep : int, optional
        States retained (default 60).

    Returns
    -------
    float
        The thermal mean-square displacement.
    """
    w = np.exp(-beta * (E[:nkeep] - E[0]))
    w = w / w.sum()
    x2n = np.array(
        [
            np.trapezoid(
                (V[:, n] / np.sqrt(np.trapezoid(V[:, n] ** 2, xs))) ** 2 * xs**2, xs
            )
            for n in range(nkeep)
        ]
    )
    return float(np.sum(w * x2n))


def ring_matrix(M, beta):
    """The oscillator ring-polymer action matrix A (section 7.20, units h=m=w=1).

    A = (M/beta)(2I - S - S^T) + (beta/M) I, a numpy.roll circulant; the
    springs stiffen as M/beta. <x^2> = (A^{-1})_00, converging on coth/2.

    Parameters
    ----------
    M : int
        Beads.
    beta : float
        Inverse temperature.

    Returns
    -------
    numpy.ndarray
        The M x M circulant.
    """
    S = np.roll(np.eye(M), 1, axis=1)
    return (M / beta) * (2.0 * np.eye(M) - S - S.T) + (beta / M) * np.eye(M)


def ring_x2(M, beta):
    """The ring-polymer bead variance (A^{-1})_00 (section 7.20, numpy.linalg.inv)."""
    return float(np.linalg.inv(ring_matrix(M, beta))[0, 0])


def staging_move(x, j, Lseg, dt, rng):
    """One staging Brownian-bridge move for the harmonic ring (section 7.21).

    Redraws Lseg-1 interior beads exactly from the spring action's Gaussian
    conditional (mean (r*prev + xR)/(r+1), variance dt*r/(r+1)); Metropolis
    accepts on the potential (1/2) x^2 alone.

    Parameters
    ----------
    x : numpy.ndarray
        Configuration, modified in place on acceptance.
    j : int
        Segment anchor.
    Lseg : int
        Segment length in links.
    dt : float
        Imaginary-time step beta/M.
    rng : numpy.random.Generator
        Seeded generator.

    Returns
    -------
    bool
        True if accepted.
    """
    M = x.size
    idxs = (j + np.arange(1, Lseg)) % M
    xL, xR = x[j], x[(j + Lseg) % M]
    prev = xL
    new = np.empty(Lseg - 1)
    xi = rng.standard_normal(Lseg - 1)
    for i in range(1, Lseg):
        r = Lseg - i
        mean = (r * prev + xR) / (r + 1)
        prev = mean + np.sqrt(dt * r / (r + 1)) * xi[i - 1]
        new[i - 1] = prev
    dS = dt * 0.5 * (float(np.sum(new**2)) - float(np.sum(x[idxs] ** 2)))
    if dS <= 0.0 or rng.random() < np.exp(-min(dS, 50.0)):
        x[idxs] = new
        return True
    return False


def pimc_x2(M, beta, nsweep, rng):
    """Staging PIMC series of the bead-averaged x^2 (section 7.21, L = M/2)."""
    dt = beta / M
    Lseg = M // 2
    x = np.zeros(M)
    mps = max(1, M // Lseg)
    series = np.empty(nsweep)
    for s in range(nsweep):
        for _ in range(mps):
            staging_move(x, int(rng.integers(M)), Lseg, dt, rng)
        series[s] = float(np.mean(x * x))
    return series


def tau_int(series, c=8.0):
    """Integrated autocorrelation time, self-consistent window (section 7.21)."""
    x = np.asarray(series, float) - np.mean(series)
    n = x.size
    acf = np.correlate(x, x, "full")[n - 1 :]
    acf = acf / acf[0]
    taus = 0.5 + np.cumsum(acf[1 : n // 2])
    W = np.arange(1, taus.size + 1)
    ok = W >= c * taus
    return float(taus[np.argmax(ok)]) if ok.any() else float(taus[-1])


def blocked_error(series, tau):
    """Blocking error bar with block length 16*tau (section 7.21)."""
    x = np.asarray(series, float)
    ell = max(1, int(np.ceil(16.0 * tau)))
    nb = max(5, x.size // ell)
    blocks = [b.mean() for b in np.array_split(x[: (x.size // nb) * nb], nb)]
    return float(np.std(blocks, ddof=1) / np.sqrt(nb))


# The grid used for every quantum face, built once (adequacy checked in Exercise 3).
XS, E_ED, V_ED = grid_ed_oscillator(1201, 10.0)
print(f"grid ED: {len(XS)} points on [-10, 10]; first levels {np.round(E_ED[:4], 5)}")

## Exercise 1 — Where the course began

The ring's near end: arithmetic is approximate, and the oscillator is where we can prove
our approximations honest. Cite {eq}`eq-ring-open`.

1. Reproduce the opening computation of [§0.1](../00-foundations/floating-point.ipynb):
   $(1 + \varepsilon) - 1 \neq \varepsilon$ for $\varepsilon = 2^{-53}$, and locate
   machine epsilon by the smallest addend $1.0$ still feels.
2. State the reading (prose): the course was the craft of controlled approximation, and
   the drift of a "conserved" energy by $\sim 10^{-11}$ is the floor
   [§0.1](../00-foundations/floating-point.ipynb) measured, not a bug.
3. State why the oscillator is the biography's subject (prose): every answer is
   closed-form, so every numerical face can be graded against truth.
4. Preview the seven faces and the spine (each face a limit of its neighbour).

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    ring_result == 0.0 and eps != 0.0 and found == eps_mach,
    "the ring's near end: (1 + eps) - 1 vanishes below machine epsilon",
    f"(1.0 + 2^-53) - 1.0 = {ring_result!r}; machine epsilon = {found:.3e}",
)

## Exercise 2 — The classical faces

Trajectory, symplectic geometry, and resonance — Volumes I–II in one degree of freedom.
Cite {eq}`eq-seven-faces`.

1. Integrate the oscillator with `scipy.integrate.solve_ivp` (DOP853, `rtol = 1e-11`) and
   verify $x(t)$ against $\cos t$; confirm the energy is conserved.
2. Contrast the velocity-Verlet update (written explicitly): its energy stays bounded
   without secular drift where forward Euler's blows up — the symplectic lesson of
   [§1.6](../01-elementary-mechanics/integrators.ipynb) and
   [§2.3](../02-classical-mechanics/hamiltonian-phase-flow.ipynb), in one contrast.
3. Build the driven–damped response and verify the resonance peak
   $\omega_d = \sqrt{\omega_0^2 - \gamma^2/2}$ (`numpy.argmax` on a fine grid against the
   closed form) and $Q = \omega_0/\gamma$, for $\gamma = 0.1$ and $0.3$ — the resonance of
   [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb).
4. Read the classical oscillator (prose): deterministic, time-reversible, resonant.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    cl_dev < 1e-8 and E_traj_spread < 1e-8,
    "the trajectory: solve_ivp meets the closed form and conserves energy",
    f"max dev {cl_dev:.1e}, energy spread {E_traj_spread:.1e}",
)
validate.check(
    verlet_spread < 1e-2 and verlet_drift < 1e-3 and euler_growth > 10.0,
    "the symplectic lesson: Verlet bounded, Euler unbounded",
    f"Verlet spread {verlet_spread:.1e}, drift {verlet_drift:.1e}; Euler grew {euler_growth:.0f}x",
)
validate.close(
    [res[0.1][0], res[0.3][0]],
    [res[0.1][1], res[0.3][1]],
    "the resonance peaks meet sqrt(w0^2 - gamma^2/2)",
    atol=1e-3,
)

## Exercise 3 — The statistical and quantum faces

Equipartition, then the ladder that breaks it. Cite {eq}`eq-seven-faces`.

1. Compute the classical $\langle x^2\rangle = kT$ by Boltzmann `scipy.integrate.quad` and
   verify at $kT = 0.5, 2.0$ — the equipartition of
   [§5.8](../05-classical-stat-mech/partition-function.ipynb).
2. Build the grid-ED oscillator (`grid_ed_oscillator`, adequacy checked against the
   thermal width) and verify the ladder $E_n = (n+\tfrac12)$ and
   $\langle 0|x^2|0\rangle = \tfrac12$ — the oscillator of
   [§6.12](../06-quantum-mechanics/harmonic-oscillator.ipynb).
3. Name the break (prose): the classical oscillator can sit still; the quantum one cannot.
4. Set up the reunion (prose): Faces 3 and 4 are the two limits of one thermal object.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    [x2_cl[0.5], x2_cl[2.0]],
    [0.5, 2.0],
    "equipartition: classical <x^2> = kT",
    rtol=1e-4,
)
validate.close(
    levels,
    np.array([0.5, 1.5, 2.5, 3.5]),
    "the quantum ladder: E_n = (n + 1/2)",
    atol=1e-2,
)
validate.close(
    x2_ground,
    0.5,
    "and the zero-point <0|x^2|0> = 1/2",
    rtol=1e-3,
)

## Exercise 4 — The thermal quantum oscillator, and the tally

The coth that contains both classical and quantum as limits — with equipartition demoted
on a measured law. Cite {eq}`eq-seven-faces`, {eq}`eq-reductions`.

1. Compute $\langle x^2\rangle_T = \tfrac12\coth(\beta/2)$ and verify against grid-ED
   thermal averages (`thermal_x2_ed`) at $\beta = 0.5, 2, 8$ — the coth of
   [§7.5](../07-quantum-statistical-mechanics/oscillator-at-temperature.ipynb).
2. Take $T \to 0$: verify $\tfrac12\coth(\beta/2) \to \tfrac12$ at $\beta = 30$ — the
   zero-point survives (Face 4 recovered).
3. Take $\hbar \to 0$ (high $T$): verify $\tfrac12\coth(\beta/2) \to kT$ and measure the
   Wigner–Kirkwood correction $(\text{quantum} - \text{classical})/\beta \to \beta/12$ at
   small $\beta$ (the law's validity window stated: leading-order).
4. Table the tally (prose plus a small table): classical laws that are really limits — $h$
   ([§7.8](../07-quantum-statistical-mechanics/classical-limit-thermal-wavelength.ipynb)),
   $\sigma$ ([§7.14](../07-quantum-statistical-mechanics/photon-gas-planck.ipynb)), $N!$
   ([§7.8](../07-quantum-statistical-mechanics/classical-limit-thermal-wavelength.ipynb)),
   the Langevin paramagnet
   ([§7.18](../07-quantum-statistical-mechanics/quantum-paramagnets-brillouin.ipynb)),
   Dulong–Petit ([§7.16](../07-quantum-statistical-mechanics/phonons-debye.ipynb)), and now
   equipartition.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    coth_dev < 1e-3,
    "the coth meets grid-ED thermal averages",
    f"max dev {coth_dev:.1e} over beta = 0.5, 2, 8",
)
validate.close(
    zero_point,
    0.5,
    "T -> 0: the zero-point survives",
    atol=1e-4,
)
validate.close(
    wk_ratio,
    1.0 / 12.0,
    "equipartition is a limit, on the measured Wigner-Kirkwood law",
    rtol=1e-3,
)

## Exercise 5 — The path-integral and sampled faces

The same thermal oscillator by imaginary time — determinant and dice. Cite
{eq}`eq-seven-faces`.

1. Build the ring-polymer determinant (`ring_x2`, reusing
   [§7.20](../07-quantum-statistical-mechanics/imaginary-time-quantum-classical.ipynb)) and
   verify $\langle x^2\rangle \to \tfrac12\coth(\beta/2)$ with the Trotter error measured
   as $1/M^2$ (the doubling ratio $\to 4$ across $M = 8$–$128$).
2. Run the staging PIMC (`pimc_x2`, the protocol of
   [§7.21](../07-quantum-statistical-mechanics/path-integral-monte-carlo.ipynb) reused
   verbatim — $L = M/2$, seeded `default_rng`, window-checked $\tau_{\mathrm{int}}$,
   blocking) and verify agreement with the exact finite-$M$ target within honest bars.
3. State what Faces 6–7 are (prose): not new oscillators but the thermal one reached by a
   different road — imaginary time instead of spectra.
4. Reflect (prose): a random walk of beads computing zero-point motion, and it is exact.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    all(3.5 < r < 4.5 for r in ratios),
    "imaginary time reaches the coth, at the 1/M^2 Trotter rate",
    f"error doubling ratios {[round(r, 2) for r in ratios]} (each ~4)",
)
validate.check(
    abs(pull) < 3.0,
    "and the sampler agrees with the exact finite-M target within bars",
    f"PIMC {mean_pimc:.4f} +/- {bar:.4f} vs exact {exact_M:.4f} ({pull:+.2f} sigma)",
)

## Exercise 6 — (STUDENT) The four-way rendezvous

One warmed oscillator's mean-square displacement, reached by mechanics, spectra, and path
integrals at once. Cite {eq}`eq-e1-rendezvous`.

1. Assemble $\langle x^2\rangle(T)$ across temperature by all four routes (ladder/spectral
   coth, grid-ED thermal average, ring-polymer determinant, classical $kT$) and plot the
   master curve with every route's points overlaid.
2. Verify the three quantum routes agree at $\beta = 2$ (to a part in $10^5$).
3. Show the classical line peeling away below the thermal scale, meeting the quantum curve
   only at high $T$ and falling short by exactly the zero-point at low $T$.
4. Say what the figure is (prose, Epilogue voice).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    [r_ladder, r_grid, r_ring],
    [r_ladder, r_ladder, r_ladder],
    "four routes, one number: the three quantum routes agree",
    atol=1e-3,
)
validate.check(
    quantum_spread < 1e-4 and r_classical < r_ladder,
    "and the classical route falls short by the zero-point",
    f"quantum spread {quantum_spread:.1e}; classical short by {r_ladder - r_classical:.4f}",
)

## Exercise 7 — (Synthesis) One object, owned

No new computation: what the biography established.

The harmonic oscillator is the most-taught system in physics because it is the one problem
everyone can solve. This course's contribution was not to solve it again but to show that
its seven textbook faces are one face seen from seven positions — and to walk between the
positions on foot, measuring the distance each time. The classical oscillator is the
quantum one with $\hbar$ sent to zero and a correction of $\beta/12$ left behind; the
ground state is the thermal one with the heat removed; the ring polymer and the Monte
Carlo walk are the thermal one reached by imaginary time instead of energy levels,
converging as one over $M$ squared. And at the centre of the biography sits a single number
— the mean-square wobble of a warmed oscillator — that Newton, Schrödinger, and Feynman
all compute, and that this notebook computed all three ways and found identical to five
digits.

That is unity of object: not a fact remembered but a thing possessed, whose every aspect
the reader can now derive from any other. It is also, quietly, the course's whole method
rehearsed on one system — closed forms to grade against, error budgets stated, independent
routes required to meet before a number is believed. Equipartition, the last classical law
the course still treated as fundamental, took its place in the tally of demotions on a
measured law; and the notebook opened where the course opened, with the admission that a
computer cannot represent $(1 + \varepsilon) - 1$, and closed by using that same fallible
arithmetic to pin a quantum wobble to five digits by three independent roads. The error
never went away. We just learned exactly how large it was — which, from the beginning, was
the whole of the craft.

The Epilogue's next step asks whether the unity runs deeper than one lucky system: whether
a single principle, not just a single object, generates the course. It does, and it has a
name — the action ([E.2](four-faces-of-the-action.ipynb)).

## Notebook summary

The Epilogue's first notebook: the oscillator's seven faces, recomputed and connected as
measured limits.

- **The ring** {eq}`eq-ring-open`: the course's first computation reproduced — $(1 +
  2^{-53}) - 1 = 0 \neq 2^{-53}$, machine epsilon located (gated) — and read forward as
  the craft of controlled approximation, the discipline the oscillator lets the course
  grade against closed forms.
- **Classical faces** {eq}`eq-seven-faces`: `solve_ivp` on $\cos t$ to $2\times10^{-11}$;
  velocity-Verlet's bounded energy against forward Euler's blow-up (the symplectic lesson
  of [§1.6](../01-elementary-mechanics/integrators.ipynb)/[§2.3](../02-classical-mechanics/hamiltonian-phase-flow.ipynb));
  the resonance peaks at $\sqrt{\omega_0^2 - \gamma^2/2}$ (all gated).
- **Statistical and quantum faces** {eq}`eq-seven-faces`: classical $\langle x^2\rangle =
  kT$ by Boltzmann quad; the ladder $E_n = (n+\tfrac12)$ and $\langle 0|x^2|0\rangle =
  \tfrac12$ by grid ED (all gated) — the zero-point break.
- **The thermal oscillator and the tally** {eq}`eq-reductions`: $\tfrac12\coth(\beta/2)$
  against grid-ED averages to $10^{-5}$; $T \to 0 \to \tfrac12$; the Wigner–Kirkwood
  $\beta/12$ verified to five digits (all gated) — equipartition demoted to a limit, the
  tally of demotions tabled.
- **Imaginary-time faces** {eq}`eq-seven-faces`: the ring determinant at the measured
  $1/M^2$ rate (doubling ratios $\to 4$, gated); the staging PIMC within one bar of the
  exact finite-$M$ target (seed 7, gated).
- **The rendezvous** {eq}`eq-e1-rendezvous`: $\langle x^2\rangle(2)$ four ways, the three
  quantum routes agreeing to a part in $10^5$ (gated), the classical line short by the
  zero-point — mechanics, spectra, and path integrals on one number.

What E.1 establishes: the course's **unity of object** — one system, fully owned, its
every face a limit of its neighbour.

## Outlook

- **[E.2](four-faces-of-the-action.ipynb) — Four Faces of the Action**: the unity climbs from a system to a principle
  ($e^{iS/\hbar} \to e^{-\beta H}$), with the double-well instanton as its summit.
- **[E.3](universality.ipynb) — Universality**: whether the microscopic details ever mattered.
- **[E.4](how-we-knew.ipynb) — How We Knew**: the method made explicit, and the ring's far end — the course's
  first computation redone under the mature protocol, its last line echoing its first.
- Cross-reference the biography's chapters
  ([§0.1](../00-foundations/floating-point.ipynb),
  [§1.2](../01-elementary-mechanics/damped-driven-pendulum.ipynb),
  [§1.6](../01-elementary-mechanics/integrators.ipynb),
  [§2.3](../02-classical-mechanics/hamiltonian-phase-flow.ipynb),
  [§5.8](../05-classical-stat-mech/partition-function.ipynb),
  [§6.12](../06-quantum-mechanics/harmonic-oscillator.ipynb),
  [§7.5](../07-quantum-statistical-mechanics/oscillator-at-temperature.ipynb),
  [§7.20](../07-quantum-statistical-mechanics/imaginary-time-quantum-classical.ipynb),
  [§7.21](../07-quantum-statistical-mechanics/path-integral-monte-carlo.ipynb)) and the
  tally of demotions
  ([§7.8](../07-quantum-statistical-mechanics/classical-limit-thermal-wavelength.ipynb),
  [§7.14](../07-quantum-statistical-mechanics/photon-gas-planck.ipynb),
  [§7.16](../07-quantum-statistical-mechanics/phonons-debye.ipynb),
  [§7.18](../07-quantum-statistical-mechanics/quantum-paramagnets-brillouin.ipynb)).

In [ ]:
from ecp.style import footer

footer()